# Fluid Dynamics

This notebook covers 4 fundamental equations of fluid dynamics, from the simplest nonlinear conservation law (Burgers) to the full Navier-Stokes system. Each demonstrates key phenomena: shock formation, viscous smoothing, Riemann problems, and shallow water waves.

Each section presents:
1. The defining equation with physical context
2. A symbolic solution
3. A numerical solution with visualization

All equations are registered in the `diff_eq_solver` equation registry.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from diff_eq_solver import registry
from diff_eq_solver.visualizer import plot_ode_solution, plot_phase_portrait, plot_pde_heatmap, plot_pde_snapshots, plot_3d_surface, plot_comparison, plot_orbit, plot_special_function
from IPython.display import Math, display

## 1. Burgers Equation

The Burgers equation is the simplest nonlinear conservation law combining advection and diffusion:

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}$$

When $\nu = 0$ (inviscid), smooth initial data develops a shock (gradient catastrophe). When $\nu > 0$ (viscous), the diffusion term prevents shock formation and smooths the solution.

**Applications:** Gas dynamics, traffic flow modeling, cosmology (structure formation), turbulence theory, acoustics.

In [ ]:
# Symbolic solution
eq = registry.get('burgers_equation')
print("Burgers Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: compare inviscid (shock) vs viscous (smooth)
Nx = 400
x = np.linspace(-2, 5, Nx)

# Initial condition: ramp function that will steepen into a shock
u0 = np.where(x < 0, 1.0, np.where(x < 1, 1.0 - x, 0.0))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
nu_values = [0.0, 0.05]
titles = ['Inviscid ($\\nu = 0$): Shock Formation', 'Viscous ($\\nu = 0.05$): Smooth Profile']

for idx, (nu, title) in enumerate(zip(nu_values, titles)):
    t_eval = np.linspace(0, 2.0, 200)
    result = eq.numerical_solve(
        t_span=(0, 2.0),
        u0=u0,
        x=x,
        t_eval=t_eval,
        params={'nu': nu}
    )
    plot_pde_heatmap(result, ax=axes[idx],
        x_label='$x$', t_label='$t$',
        title=f'Burgers: {title}')

plt.suptitle('Burgers Equation: Inviscid vs Viscous', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Snapshot comparison at several times
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for idx, (nu, title) in enumerate(zip(nu_values, titles)):
    t_eval = np.linspace(0, 2.0, 200)
    result = eq.numerical_solve(
        t_span=(0, 2.0),
        u0=u0,
        x=x,
        t_eval=t_eval,
        params={'nu': nu}
    )
    plot_pde_snapshots(result, ax=axes[idx],
        x_label='$x$', y_label='$u(x,t)$',
        title=f'Burgers: {title}', num_snapshots=6)

plt.tight_layout()
plt.show()

## 2. Navier-Stokes (1D)

The 1D Navier-Stokes equation for compressible flow with viscosity:

$$\rho\left(\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x}\right) = -\frac{\partial p}{\partial x} + \mu \frac{\partial^2 u}{\partial x^2}$$

Together with continuity: $\frac{\partial \rho}{\partial t} + \frac{\partial (\rho u)}{\partial x} = 0$ and an equation of state $p = p(\rho)$.

**Applications:** Aerodynamics, pipe flow, blood flow, ocean currents, atmospheric dynamics, industrial fluid processing.

In [ ]:
# Symbolic solution
eq = registry.get('navier_stokes_1d')
print("Navier-Stokes (1D):")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: viscous flow evolution
Nx = 300
x = np.linspace(-5, 5, Nx)
t_eval = np.linspace(0, 3.0, 200)

# Initial velocity profile: Gaussian bump
u0 = 2.0 * np.exp(-x**2 / 2.0)
rho0 = np.ones(Nx)  # uniform density initially

result = eq.numerical_solve(
    t_span=(0, 3.0),
    u0=np.column_stack([rho0, u0]),
    x=x,
    t_eval=t_eval,
    params={'mu': 0.1, 'gamma_eos': 1.4}
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Velocity heatmap
plot_pde_heatmap(result, ax=axes[0],
    x_label='$x$', t_label='$t$',
    title='Navier-Stokes: Velocity $u(x,t)$')

# Velocity snapshots
plot_pde_snapshots(result, ax=axes[1],
    x_label='$x$', y_label='$u(x,t)$',
    title='Navier-Stokes: Velocity Snapshots', num_snapshots=6)

plt.suptitle('1D Navier-Stokes: Viscous Flow ($\\mu = 0.1$)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Euler Equations (1D)

The 1D Euler equations for inviscid compressible flow are a system of conservation laws:

$$\frac{\partial \rho}{\partial t} + \frac{\partial (\rho u)}{\partial x} = 0$$

$$\frac{\partial (\rho u)}{\partial t} + \frac{\partial (\rho u^2 + p)}{\partial x} = 0$$

$$\frac{\partial E}{\partial t} + \frac{\partial ((E + p)u)}{\partial x} = 0$$

with total energy $E = \frac{p}{\gamma - 1} + \frac{1}{2}\rho u^2$.

**Applications:** Sod shock tube problem, supersonic flow, explosion modeling, astrophysical gas dynamics, nozzle design.

In [ ]:
# Symbolic solution
eq = registry.get('euler_equations_1d')
print("Euler Equations (1D):")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: Sod shock tube problem
# Classic test: diaphragm separates left (high pressure) and right (low pressure) gas

Nx = 400
x = np.linspace(0, 1, Nx)
t_eval = np.linspace(0, 0.2, 200)

# Sod initial conditions (gamma = 1.4)
gamma = 1.4
rho_L, u_L, p_L = 1.0, 0.0, 1.0    # left state
rho_R, u_R, p_R = 0.125, 0.0, 0.1  # right state

rho0 = np.where(x < 0.5, rho_L, rho_R)
u0 = np.where(x < 0.5, u_L, u_R)
p0 = np.where(x < 0.5, p_L, p_R)
E0 = p0 / (gamma - 1) + 0.5 * rho0 * u0**2

result = eq.numerical_solve(
    t_span=(0, 0.2),
    u0=np.column_stack([rho0, rho0 * u0, E0]),
    x=x,
    t_eval=t_eval,
    params={'gamma': gamma}
)

# Extract density, velocity, pressure at final time
rho_final = result.u[-1, :Nx]
momentum_final = result.u[-1, Nx:2*Nx]
E_final = result.u[-1, 2*Nx:]
u_final = momentum_final / rho_final
p_final = (gamma - 1) * (E_final - 0.5 * rho_final * u_final**2)

# Plot density, velocity, pressure profiles
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(x, rho_final, 'b-', linewidth=2)
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$\\rho$')
axes[0].set_title('Density $\\rho(x)$')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, u_final, 'r-', linewidth=2)
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('$u$')
axes[1].set_title('Velocity $u(x)$')
axes[1].grid(True, alpha=0.3)

axes[2].plot(x, p_final, 'g-', linewidth=2)
axes[2].set_xlabel('$x$')
axes[2].set_ylabel('$p$')
axes[2].set_title('Pressure $p(x)$')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Sod Shock Tube Problem (Euler Equations, $t = 0.2$)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of density evolution showing shock, contact, and rarefaction
fig, ax = plt.subplots(figsize=(10, 6))

# Re-solve and plot density heatmap
result_full = eq.numerical_solve(
    t_span=(0, 0.2),
    u0=np.column_stack([rho0, rho0 * u0, E0]),
    x=x,
    t_eval=t_eval,
    params={'gamma': gamma}
)

# Extract density field
rho_field = result_full.u[:, :Nx]
ax.pcolormesh(x, t_eval, rho_field, shading='auto', cmap='viridis')
ax.set_xlabel('$x$')
ax.set_ylabel('$t$')
ax.set_title('Sod Shock Tube: Density Evolution $\\rho(x,t)$')
plt.colorbar(ax.pcolormesh(x, t_eval, rho_field, shading='auto', cmap='viridis'),
             ax=ax, label='$\\rho$')
plt.tight_layout()
plt.show()

## 4. Shallow Water Equations

The 1D shallow water equations (Saint-Venant equations) describe flow in a channel:

$$\frac{\partial h}{\partial t} + \frac{\partial (hu)}{\partial x} = 0$$

$$\frac{\partial (hu)}{\partial t} + \frac{\partial}{\partial x}\left(hu^2 + \frac{1}{2}gh^2\right) = 0$$

where $h$ is the water height, $u$ is the depth-averaged velocity, and $g$ is gravitational acceleration.

**Applications:** Dam break analysis, tsunami propagation, tidal flows, flood modeling, storm surge prediction, irrigation channels.

In [ ]:
# Symbolic solution
eq = registry.get('shallow_water_equations')
print("Shallow Water Equations:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: dam break problem
# A vertical wall of water is released at t=0

Nx = 400
x = np.linspace(0, 1, Nx)
t_eval = np.linspace(0, 0.3, 200)
g = 9.81

# Dam break IC: high water on left, low water on right
h_L = 1.0   # water height on left
h_R = 0.1   # water height on right
dam_pos = 0.3  # dam position

h0 = np.where(x < dam_pos, h_L, h_R)
hu0 = np.zeros(Nx)  # initially at rest

result = eq.numerical_solve(
    t_span=(0, 0.3),
    u0=np.column_stack([h0, hu0]),
    x=x,
    t_eval=t_eval,
    params={'g': g}
)

# Extract height and velocity fields
h_field = result.u[:, :Nx]
hu_field = result.u[:, Nx:]
# Avoid division by zero
u_field = np.where(h_field > 1e-10, hu_field / h_field, 0.0)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Height heatmap
im0 = axes[0, 0].pcolormesh(x, t_eval, h_field, shading='auto', cmap='Blues')
axes[0, 0].set_xlabel('$x$')
axes[0, 0].set_ylabel('$t$')
axes[0, 0].set_title('Water Height $h(x,t)$')
plt.colorbar(im0, ax=axes[0, 0], label='$h$')

# Velocity heatmap
im1 = axes[0, 1].pcolormesh(x, t_eval, u_field, shading='auto', cmap='Reds')
axes[0, 1].set_xlabel('$x$')
axes[0, 1].set_ylabel('$t$')
axes[0, 1].set_title('Velocity $u(x,t)$')
plt.colorbar(im1, ax=axes[0, 1], label='$u$')

# Height snapshots
snapshot_indices = [0, len(t_eval)//5, 2*len(t_eval)//5, 3*len(t_eval)//5, 4*len(t_eval)//5]
for si in snapshot_indices:
    axes[1, 0].plot(x, h_field[si, :], linewidth=1.5,
        label=f'$t = {t_eval[si]:.3f}$')
axes[1, 0].set_xlabel('$x$')
axes[1, 0].set_ylabel('$h(x,t)$')
axes[1, 0].set_title('Dam Break: Height Snapshots')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Velocity snapshots
for si in snapshot_indices:
    axes[1, 1].plot(x, u_field[si, :], linewidth=1.5,
        label=f'$t = {t_eval[si]:.3f}$')
axes[1, 1].set_xlabel('$x$')
axes[1, 1].set_ylabel('$u(x,t)$')
axes[1, 1].set_title('Dam Break: Velocity Snapshots')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Shallow Water Equations: Dam Break Evolution', fontsize=14)
plt.tight_layout()
plt.show()